In [1]:
import numpy as np
import scipy.stats as sps
import pandas as pd

# Задаем параметры
theta = 5.0
n = 100
beta = 0.95  

# Генерируем выборку из равномерного распределения U[theta, 2*theta]
sample = np.random.uniform(low=theta, high=2 * theta, size=n)

print(f"Значение theta: {theta}")
print(f"Размер выборки: {n}")
print(f"Первые 5 элементов выборки: {sample[:5].round(3)}")

Значение theta: 5.0
Размер выборки: 100
Первые 5 элементов выборки: [6.961 9.316 8.803 9.848 7.448]


In [2]:
# f) Точный доверительный интервал
max_x = sample.max()

t1 = ((1 - beta) / 2) ** (1 / n)
t2 = ((1 + beta) / 2) ** (1 / n)

left = max_x / (1 + t2)
right = max_x / (1 + t1)

print(f"Точный ДИ: ({left:.4f}, {right:.4f})")
print(f"Длина точного ДИ: {right - left:.4f}")

Точный ДИ: (4.9647, 5.0556)
Длина точного ДИ: 0.0909


In [3]:
# f) Асимптотический доверительный интервал (на базе ОММ)
mean_x = sample.mean()
s = np.std(sample, ddof=1) 
z = sps.norm.ppf((1 + beta) / 2)

omm_est = (2 / 3) * mean_x
coef = z * 2 * s * np.sqrt(n - 1) / (3 * n)

asymp_left = omm_est - coef
asymp_right = omm_est + coef

print(f"Асимптотический ДИ: ({asymp_left:.4f}, {asymp_right:.4f})")
print(f"Длина асимптотического ДИ: {asymp_right - asymp_left:.4f}")

Асимптотический ДИ: (4.7519, 5.1505)
Длина асимптотического ДИ: 0.3986


In [4]:
B = 1000 

theta_omp = (n + 1) / (2 * n + 1) * sample.max()

bootstrap_deltas = []

for _ in range(B):
    boot_sample = np.random.choice(sample, size=n, replace=True)
    
    boot_est = (n + 1) / (2 * n + 1) * boot_sample.max()
    
    delta = boot_est - theta_omp
    bootstrap_deltas.append(delta)

delta_lower_quantile = np.percentile(bootstrap_deltas, (1 - beta) / 2 * 100)
delta_upper_quantile = np.percentile(bootstrap_deltas, (1 + beta) / 2 * 100)

boot_left = theta_omp - delta_upper_quantile
boot_right = theta_omp - delta_lower_quantile

print(f"Бутстраповский ДИ: ({boot_left:.4f}, {boot_right:.4f})")
print(f"Длина бутстраповского ДИ: {boot_right - boot_left:.4f}")

Бутстраповский ДИ: (4.9888, 5.0288)
Длина бутстраповского ДИ: 0.0401


In [5]:
param_bootstrap_deltas = []
B = 50_000

for _ in range(B):
    boot_sample_param = np.random.uniform(low=theta_omp, high=2 * theta_omp, size=n)
    
    boot_est_param = (n + 1) / (2 * n + 1) * boot_sample_param.max()
    
    delta_param = boot_est_param - theta_omp
    param_bootstrap_deltas.append(delta_param)

param_delta_lower_quantile = np.percentile(param_bootstrap_deltas, (1 - beta) / 2 * 100)
param_delta_upper_quantile = np.percentile(param_bootstrap_deltas, (1 + beta) / 2 * 100)

param_boot_left = theta_omp - param_delta_upper_quantile
param_boot_right = theta_omp - param_delta_lower_quantile

print(f"Параметр. бутстраповский ДИ: ({param_boot_left:.4f}, {param_boot_right:.4f})")
print(f"Длина параметр. бутстраповского ДИ: {param_boot_right - param_boot_left:.4f}")

Параметр. бутстраповский ДИ: (4.9646, 5.0547)
Длина параметр. бутстраповского ДИ: 0.0901


In [6]:
# h) Сравнить все интервалы
results = {
    "Метод": [
        "Точный",
        "Асимптотический (на базе ОММ)",
        "Непараметрический бутстрап (на базе испр. ОМП)",
        "Параметрический бутстрап (на базе испр. ОМП)"
    ],
    "Левая граница": [left, asymp_left, boot_left, param_boot_left],
    "Правая граница": [right, asymp_right, boot_right, param_boot_right],
    "Длина": [
        right - left, 
        asymp_right - asymp_left, 
        boot_right - boot_left,
        param_boot_right - param_boot_left
    ]
}

df = pd.DataFrame(results)

df[f"Накрыл theta={theta}"] = (df["Левая граница"] <= theta) & (df["Правая граница"] >= theta)

df.round(4)

,Метод,Левая граница,Правая граница,Длина,Накрыл theta=5.0
0,Точный,4.9647,5.0556,0.0909,True
1,Асимптотический (на базе ОММ),4.7519,5.1505,0.3986,True
2,Непараметрический бутстрап (на базе испр. ОМП),4.9888,5.0288,0.0401,True
3,Параметрический бутстрап (на базе испр. ОМП),4.9646,5.0547,0.0901,True
